> **Chapter 15, Part 5** | Engineering lens. **Focus:** when an asset fails, skip its descendants rather than build them on missing inputs; retry transient failures; and quantify the damage with the Chapter 12 blast radius.

# Failure, Retries, and Blast Radius

Everything so far assumed the happy path. Real orchestration is mostly about the unhappy path. Three questions:

1. When `clean` fails, what happens to `rollup` downstream? It must be **skipped**, not built on a missing input. Building on missing data is how a failed run silently produces a wrong dashboard.
2. Some failures are transient (a network blip). A **retry policy** gives an asset N attempts before it is declared failed.
3. When an asset fails, how much breaks? That is its **blast radius**, the set of downstream descendants, the exact descriptor Chapter 12 built for lineage risk.

In [1]:
# The tiny orchestrator we build across 15.1-15.3, collected into one cell.
from collections import deque
from dataclasses import dataclass
from typing import Callable, Optional


@dataclass
class Asset:
    name: str
    deps: list
    compute: Callable
    partitioned: bool = False


class AssetGraph:
    def __init__(self):
        self.assets = {}

    def add(self, asset):
        self.assets[asset.name] = asset
        return self

    def _downstream(self):
        down = {n: [] for n in self.assets}
        for n, a in self.assets.items():
            for d in a.deps:
                if d in down:
                    down[d].append(n)
        return down

    def topological_order(self):
        indeg = {n: 0 for n in self.assets}
        for n, a in self.assets.items():
            for d in a.deps:
                if d in indeg:
                    indeg[n] += 1
        down = self._downstream()
        q = deque(sorted(n for n, k in indeg.items() if k == 0))
        order = []
        while q:
            n = q.popleft()
            order.append(n)
            for m in sorted(down[n]):
                indeg[m] -= 1
                if indeg[m] == 0:
                    q.append(m)
        if len(order) != len(self.assets):
            stuck = sorted(n for n in self.assets if n not in order)
            raise ValueError("cycle detected among assets: " + ", ".join(stuck))
        return order

    def _needed(self, targets):
        need = set()
        stack = list(targets)
        while stack:
            n = stack.pop()
            if n in need or n not in self.assets:   # ignore refs outside the graph
                continue
            need.add(n)
            stack.extend(self.assets[n].deps)
        return need

    def materialize(self, targets=None, log=None, partition=None, verbose=True):
        order = self.topological_order()
        if targets is not None:
            need = self._needed(targets)
            order = [n for n in order if n in need]
        results = {}
        for n in order:
            tag = " [" + str(partition) + "]" if partition is not None else ""
            if log is not None and log.is_materialized(n, partition):
                results[n] = log.value(n, partition)
                if verbose:
                    print("  skip   " + n + tag + " (already materialized)")
                continue
            inputs = {d: results.get(d) for d in self.assets[n].deps}
            value = self.assets[n].compute(inputs)
            results[n] = value
            if log is not None:
                log.record(n, partition, value)
            if verbose:
                print("  build  " + n + tag)
        return results


class MaterializationLog:
    def __init__(self):
        self.store = {}

    def is_materialized(self, name, partition=None):
        return (name, partition) in self.store

    def record(self, name, partition=None, value=None):
        self.store[(name, partition)] = value

    def value(self, name, partition=None):
        return self.store[(name, partition)]


def backfill(graph, target, partitions, log, verbose=True):
    report = {"materialized": [], "skipped": []}
    order = [n for n in graph.topological_order() if n in graph._needed([target])]
    for p in partitions:
        for n in order:
            if log.is_materialized(n, p):
                report["skipped"].append((n, p))
                continue
            inputs = {d: (log.value(d, p) if log.is_materialized(d, p) else None)
                      for d in graph.assets[n].deps}
            log.record(n, p, graph.assets[n].compute(inputs))
            report["materialized"].append((n, p))
    return report


def blast_radius(graph, failed):
    down = graph._downstream()
    seen, stack = set(), list(down[failed])
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n)
        stack.extend(down[n])
    return seen


@dataclass
class RetryPolicy:
    max_attempts: int = 1


def materialize_with_failures(graph, failing=None, retry=None, verbose=True):
    failing = failing or {}
    retry = retry or RetryPolicy()
    order = graph.topological_order()
    status, results, attempts_used = {}, {}, {}
    for n in order:
        if any(status.get(d) in ("failed", "skipped") for d in graph.assets[n].deps):
            status[n] = "skipped"
            if verbose:
                print("  skip    " + n + " (upstream failed)")
            continue
        attempts, ok = 0, False
        while attempts < retry.max_attempts:
            attempts += 1
            if attempts <= failing.get(n, 0):
                if verbose:
                    print("  retry   " + n + " attempt " + str(attempts) + " failed")
                continue
            ok = True
            break
        attempts_used[n] = attempts
        if ok:
            inputs = {d: results.get(d) for d in graph.assets[n].deps}
            results[n] = graph.assets[n].compute(inputs)
            status[n] = "materialized"
            if verbose:
                print("  build   " + n + " (attempt " + str(attempts) + ")")
        else:
            status[n] = "failed"
            if verbose:
                print("  FAIL    " + n + " (exhausted " + str(retry.max_attempts) + " attempts)")
    return status, results, attempts_used


print("orchestrator core ready:", len([Asset, AssetGraph, MaterializationLog,
      backfill, blast_radius, RetryPolicy, materialize_with_failures]), "building blocks")

orchestrator core ready: 7 building blocks


## Skip-on-failure

Build a diamond: `ingest` feeds both `clean_a` and `clean_b`, which both feed `join`, which feeds `report`. Force `clean_a` to fail every attempt. Watch `join` and `report` get skipped, while `clean_b` still builds (it does not depend on the failure).

In [2]:
g = AssetGraph()
g.add(Asset("ingest", [], lambda i: 1))
g.add(Asset("clean_a", ["ingest"], lambda i: 1))
g.add(Asset("clean_b", ["ingest"], lambda i: 1))
g.add(Asset("join", ["clean_a", "clean_b"], lambda i: 1))
g.add(Asset("report", ["join"], lambda i: 1))

status, _, _ = materialize_with_failures(g, failing={"clean_a": 99})
print()
print("final status:")
for n, s in status.items():
    print(f"  {n:9s} {s}")

  build   ingest (attempt 1)
  retry   clean_a attempt 1 failed
  FAIL    clean_a (exhausted 1 attempts)
  build   clean_b (attempt 1)
  skip    join (upstream failed)
  skip    report (upstream failed)

final status:
  ingest    materialized
  clean_a   failed
  clean_b   materialized
  join      skipped
  report    skipped


## Validation V004: nothing downstream of a failure is materialized

The guarantee is that no asset is ever built on a failed or missing input. We assert it on the run above.

In [3]:
skipped = {n for n, s in status.items() if s == "skipped"}
materialized = {n for n, s in status.items() if s == "materialized"}
assert "join" in skipped and "report" in skipped, "downstream of failure must skip"
assert "clean_b" in materialized, "siblings of a failure still build"
print("V004 holds: join and report skipped, clean_b still built")

V004 holds: join and report skipped, clean_b still built


## Retries turn a transient failure into a success

Now let `clean_a` fail only twice, then succeed. A retry policy of three attempts absorbs it; the whole graph completes.

In [4]:
status2, _, attempts = materialize_with_failures(
    g, failing={"clean_a": 2}, retry=RetryPolicy(max_attempts=3))
print()
print(f"clean_a took {attempts['clean_a']} attempts; final status of report:", status2["report"])

  build   ingest (attempt 1)
  retry   clean_a attempt 1 failed
  retry   clean_a attempt 2 failed
  build   clean_a (attempt 3)
  build   clean_b (attempt 1)
  build   join (attempt 1)
  build   report (attempt 1)

clean_a took 3 attempts; final status of report: materialized


## Validation V005: blast radius equals downstream descendants

The blast radius of a failed asset is everything reachable from it in the DAG. We compute it with our own traversal and cross-check against NetworkX's `descendants`.

In [5]:
import networkx as nx

G = nx.DiGraph()
for n, a in g.assets.items():
    for d in a.deps:
        G.add_edge(d, n)

for failed in ("ingest", "clean_a", "report"):
    ours = blast_radius(g, failed)
    theirs = nx.descendants(G, failed)
    assert ours == theirs, f"mismatch for {failed}"
    print(f"  fail {failed:9s} -> blast radius {len(ours)}: {sorted(ours) or '(none, it is a leaf)'}")

print()
print("V005 holds: blast radius equals graph descendants")

  fail ingest    -> blast radius 4: ['clean_a', 'clean_b', 'join', 'report']
  fail clean_a   -> blast radius 2: ['join', 'report']
  fail report    -> blast radius 0: (none, it is a leaf)

V005 holds: blast radius equals graph descendants


Blast radius is the number that should decide your on-call priority. A failure in `ingest` breaks everything; a failure in `report` breaks nothing downstream. Chapter 12 ranked stewardship by exactly this descriptor. Orchestration is where it earns its keep. Next: map all of this onto Dagster.